# Lab22 — Chạy tiếp trên Kaggle

Upload notebook này lên Kaggle. Chọn GPU và bật Internet. Thêm private Dataset có `Lab22_continue.zip`, gồm adapter trọng số và kết quả từ Drive. Hoặc điền HF_DATASET_ID để tải trực tiếp từ repo HF dataset riêng tư, thêm Secret HF_TOKEN trên Kaggle. Zip bài nộp không đủ vì không chứa trọng số adapter. Chạy một phần mỗi lần. Đây là bản chuẩn bị, chưa kiểm chứng GPU Kaggle.

In [ ]:
ONLY = "variants-recovery"  # hoặc "benchmark", "grpo", "beta", "gguf", "finish"
HF_DATASET_ID = ""  # tài-khoản/lab22-continue; ưu tiên HF nếu điền
INPUT_ZIP = ""  # để trống để tìm Lab22_continue.zip dưới /kaggle/input

In [ ]:
import os, sys, json, gzip, base64, zipfile, subprocess
from pathlib import Path
WORK = Path("/kaggle/working/lab22")
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / "adapters/sft-mini/adapter_model.safetensors").exists():
    if HF_DATASET_ID:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
        from huggingface_hub import snapshot_download
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        snapshot_download(repo_id=HF_DATASET_ID, repo_type="dataset", token=hf_token, local_dir=str(WORK))
        del hf_token
    else:
        candidates = [Path(INPUT_ZIP)] if INPUT_ZIP else list(Path("/kaggle/input").rglob("Lab22_continue.zip"))
        assert len(candidates) == 1, "Thêm Dataset chứa Lab22_continue.zip hoặc điền INPUT_ZIP"
        with zipfile.ZipFile(candidates[0]) as z:
            for name in z.namelist():
                path = (WORK / name).resolve()
                assert path == WORK or WORK in path.parents, "Đường dẫn zip không hợp lệ"
            z.extractall(WORK)
payload = json.loads(gzip.decompress(base64.b64decode("...")))
for relative, contents in payload.items():
    if relative.startswith("submission/"): continue
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding="utf-8")
assert (WORK / "adapters/sft-mini/adapter_model.safetensors").exists(), "Zip thiếu adapter SFT"
os.chdir(WORK)
# Dùng 1 GPU cho pipeline hiện tại, kể cả máy Kaggle có 2 GPU.
os.environ.update(CUDA_VISIBLE_DEVICES="0", COMPUTE_TIER="T4", GEN_BATCH_SIZE="2", JUDGE_PROVIDER="rm", STUDENT_NAME="Phạm Hoàng Trọng", STUDENT_COHORT="A20-K4 · 2A202602765")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['unsloth>=2026.10.1', 'trl>=1.13,<1.14', 'transformers>=5.2,<5.18', 'peft>=0.18,<1.0', 'accelerate>=1.10,<2.0', 'bitsandbytes>=0.48,<1.0', 'datasets>=4.7,<5.0', 'matplotlib>=3.9,<4.0', 'pandas>=2.2,<4.0', 'pyarrow>=17', 'openai>=1.55,<4.0', 'anthropic>=0.40,<2.0'], "nbclient>=0.10,<1", "nbformat>=5.10,<6", "ipykernel>=6,<8"], check=True)
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Chọn GPU trong Settings'; print(torch.cuda.get_device_name(0))"], check=True)
subprocess.run([sys.executable, "-u", "scripts/local_model_storage.py", "--root", str(WORK), "--local", "/kaggle/temp/lab22-runtime"], check=True)

In [ ]:
if ONLY == "variants-recovery":
    for name in ("dpo", "rpo"):
        assert (WORK / "adapters/variants" / name / "adapter_model.safetensors").exists(), f"Zip thiếu adapter variants/{name}"
    command = [sys.executable, "-u", "scripts/restore_variant_metrics.py", "--root", str(WORK)]
else:
    command = [sys.executable, "-u", "scripts/run_colab.py", "--only", ONLY]
result = subprocess.run(command)
print("Exit code:", result.returncode)
status = WORK / "data/eval/run_status.json"
if status.exists():
    print({k:v["status"] for k,v in json.loads(status.read_text()).items()})
if ONLY == "variants-recovery" and result.returncode != 0:
    raise RuntimeError("Khôi phục variants thất bại; xem traceback phía trên. Chưa cập nhật đủ 5 dòng.")

In [ ]:
from IPython.display import FileLink, display
sys.path.insert(0, str(WORK / "scripts"))
from run_colab import export_submission
archive = export_submission()
display(FileLink(str(archive)))
print("Tải zip về, giải nén vào repo rồi commit/push. Muốn chạy phiên mới, giữ cả adapter bằng cách tải continuation zip dưới đây.")
continue_zip = Path("/kaggle/working/Lab22_continue.zip")
with zipfile.ZipFile(continue_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("adapters", "data", "notebooks", "submission"):
        for p in (WORK / folder).rglob("*"):
            if p.is_file() and p.suffix in (".safetensors", ".json", ".jsonl", ".parquet", ".ipynb", ".md", ".txt", ".png", ".jpg", ".jinja", ".model"):
                z.write(p, p.relative_to(WORK).as_posix())
display(FileLink(str(continue_zip)))

Trước khi dừng, tải zip và lưu phiên notebook/output theo chức năng Save Version của Kaggle. Mô hình lớn trong /kaggle/temp không phải bằng chứng nộp bài; phiên mới khôi phục từ adapter. Đánh giá đang dở chưa resume ở cấp câu hỏi; benchmark giữ các lượt hoàn thành có cấu hình khớp. Giữ Dataset riêng tư vì chứa adapter/dữ liệu thực nghiệm.